# Wave Artifact 분석 & 시뮬레이션

EM 이미지의 금속 분포 간섭으로 생기는 **파동(ripple) 밝기 artifact**를 분석하고, 그것을
**band-pass noise 기반 wave augmentation**으로 시뮬레이션해 시각 검증하는 노트북.

흐름:
1. 이미지 로딩 (wave 있는 그룹 / 없는 그룹)
2. **FFT 측정** — radial / angular power profile, 그룹 차이로 ripple band 추정
3. band 설정 (auto 추정 + 수동 override)
4. **band-pass noise wave field** generator
5. **wave augmentation** (mean-preserving 곱셈 / log 도메인, float 유지 → saturation 없음)
6. 시각 검증 (before/after, histogram, FFT band 매칭)
7. (선택) FFT notch de-wave

설계 근거: `docs/ANALYSIS_grayscale_separation.md` §5–7. 의존성: numpy, Pillow, matplotlib (torch 불필요).

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ── 설정: 이미지 폴더 경로 ─────────────────────────────────────────
# wave 가 보이는 이미지들 / 안 보이는 이미지들을 나눠 두면 그룹 차이로 band 추정이 깔끔.
# 그룹을 못 나누면 WITHOUT_WAVE_DIR = None 로 두고 WITH 만으로 eyeball.
WITH_WAVE_DIR    = r"D:\path\to\images_with_wave"      # <-- 채우기
WITHOUT_WAVE_DIR = r"D:\path\to\images_without_wave"   # <-- 없으면 None

MAX_IMAGES_PER_GROUP = 30   # 평균 spectrum 계산에 쓸 최대 장수
RNG = np.random.default_rng(0)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

## 1. 이미지 로딩

In [ ]:
def list_images(folder, limit=None):
    if folder is None:
        return []
    paths = sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMG_EXTS)
    return paths[:limit] if limit else paths


def load_gray(path):
    """grayscale float [0,1] 로 로드."""
    im = Image.open(path)
    arr = np.asarray(im)
    if arr.ndim == 3:
        arr = arr[..., 0]            # mask/RGB 면 첫 채널 (dataset.py 와 동일 규약)
    arr = arr.astype(np.float32)
    if arr.max() > 1.0:
        arr = arr / 255.0
    return arr


with_paths = list_images(WITH_WAVE_DIR, MAX_IMAGES_PER_GROUP)
without_paths = list_images(WITHOUT_WAVE_DIR, MAX_IMAGES_PER_GROUP)
print(f"with-wave  : {len(with_paths)} images")
print(f"without    : {len(without_paths)} images")

assert with_paths, "WITH_WAVE_DIR 에 이미지가 없습니다. 경로를 확인하세요."
sample = load_gray(with_paths[0])
print("sample shape", sample.shape, "range", float(sample.min()), float(sample.max()))
plt.figure(figsize=(4, 4)); plt.imshow(sample, cmap="gray"); plt.title("sample (with wave)"); plt.axis("off"); plt.show()

## 2. FFT 측정

- **radial power profile** $P(k)$: 방위각 평균. ripple 은 특정 반경(주파수)에서 peak.
- 주파수 단위는 **cycles/pixel** (`fftfreq`) → 이미지 크기 무관 → 여러 장 평균 가능.
- **그룹 차이** $P_{\text{with}}(k)-P_{\text{without}}(k)$ 가 ripple band 를 드러냄.

In [ ]:
def power_spectrum(img):
    """DC 제거 후 shift 된 2D power spectrum + 주파수 좌표."""
    H, W = img.shape
    F = np.fft.fftshift(np.fft.fft2(img - img.mean()))
    P = np.abs(F) ** 2
    fy = np.fft.fftshift(np.fft.fftfreq(H))
    fx = np.fft.fftshift(np.fft.fftfreq(W))
    KX, KY = np.meshgrid(fx, fy)
    return P, KX, KY


def radial_profile(img, nbins=128):
    P, KX, KY = power_spectrum(img)
    K = np.sqrt(KX ** 2 + KY ** 2).ravel()
    Pr = P.ravel()
    bins = np.linspace(0, 0.5, nbins + 1)
    idx = np.digitize(K, bins) - 1
    valid = (idx >= 0) & (idx < nbins)
    sums = np.bincount(idx[valid], weights=Pr[valid], minlength=nbins)
    cnts = np.bincount(idx[valid], minlength=nbins)
    prof = sums / np.maximum(cnts, 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    return centers, prof


def avg_radial(paths, nbins=128):
    accs = []
    centers = None
    for p in paths:
        c, prof = radial_profile(load_gray(p), nbins)
        centers = c
        # 장마다 에너지 스케일 차이 → 총합 정규화 후 평균 (형태 비교 목적)
        accs.append(prof / (prof.sum() + 1e-12))
    return centers, np.mean(accs, axis=0)


# 단일 이미지 spectrum 시각화
P, KX, KY = power_spectrum(sample)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(sample, cmap="gray"); plt.title("image"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(np.log1p(P), cmap="magma")
plt.title("log power spectrum (shifted)"); plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 그룹 radial profile 비교 → ripple band
# 주의: 물질 경계(DC step)의 edge 는 강한 저주파 에너지를 만들어 단일 그룹 peak 를 속인다.
#       → without 그룹과의 diff 가 edge 를 상쇄해 ripple band 만 남기므로 가장 신뢰도 높음.
centers, prof_with = avg_radial(with_paths)
plt.figure(figsize=(8, 4.5))
plt.plot(centers, prof_with, label="with wave", lw=2)

if without_paths:
    _, prof_without = avg_radial(without_paths)
    plt.plot(centers, prof_without, label="without wave", lw=2)
    diff = prof_with - prof_without
    plt.plot(centers, diff, label="diff (with - without)", lw=1.5, ls="--")
    peak_k = centers[np.argmax(diff)]
    print(f"diff peak frequency  k* = {peak_k:.4f} cycles/pixel  (파장 ~ {1/peak_k:.1f} px)")
else:
    prof_without = None
    # 단일 그룹: edge 의 저주파 에너지를 피하려 LO_FLOOR 아래는 무시 (그래도 부정확).
    LO_FLOOR = 0.02
    m = centers >= LO_FLOOR
    peak_k = centers[m][np.argmax(prof_with[m])]
    print(f"(without 그룹 없음 — 추정 부정확) with-wave peak  k* = {peak_k:.4f} cyc/px  (파장 ~ {1/peak_k:.1f} px)")
    print("  ↑ edge 영향으로 신뢰도 낮음. 가능하면 WITHOUT_WAVE_DIR 를 채워 diff 로 측정하거나,")
    print("    아래 §3 에서 K_LO/K_HI 를 power-spectrum plot 보고 직접 조정하세요.")

plt.axvline(peak_k, color="k", alpha=0.3)
plt.xlabel("radial frequency (cycles/pixel)"); plt.ylabel("normalized power")
plt.title("radial power profile"); plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
# 방향성(angular) 확인 — peak_k 주변 band 에서 각도별 power
def angular_profile(img, k_lo, k_hi, nbins=90):
    P, KX, KY = power_spectrum(img)
    K = np.sqrt(KX ** 2 + KY ** 2)
    ang = (np.degrees(np.arctan2(KY, KX)) % 180)
    band = (K >= k_lo) & (K <= k_hi)
    a = ang[band].ravel(); w = P[band].ravel()
    bins = np.linspace(0, 180, nbins + 1)
    idx = np.clip(np.digitize(a, bins) - 1, 0, nbins - 1)
    sums = np.bincount(idx, weights=w, minlength=nbins)
    cnts = np.bincount(idx, minlength=nbins)
    return 0.5 * (bins[:-1] + bins[1:]), sums / np.maximum(cnts, 1)

band_half = 0.6 * peak_k
k_lo_est, k_hi_est = max(peak_k - band_half, 1e-3), peak_k + band_half
angs, ap = angular_profile(sample, k_lo_est, k_hi_est)
plt.figure(figsize=(7, 3.5))
plt.plot(angs, ap); plt.xlabel("angle (deg)"); plt.ylabel("power in band")
plt.title(f"angular profile in band [{k_lo_est:.3f}, {k_hi_est:.3f}]"); plt.grid(alpha=0.3); plt.show()
print("방향 peak (deg):", float(angs[np.argmax(ap)]), "| 평평하면 등방성(방향 무관)")

## 3. band 설정 (auto 추정 + 수동 override)

위 radial/angular plot 을 보고 band 를 확정한다. auto 추정값을 기본으로 두되, plot 이 더
넓/좁은 band 를 보이면 직접 조정. `DIRECTION=None` 이면 등방성(전방향) wave.

In [ ]:
# ── 측정에서 읽은 값으로 조정 ──────────────────────────────────────
K_LO = float(k_lo_est)     # ripple band 하한 (cycles/pixel)
K_HI = float(k_hi_est)     # ripple band 상한
DIRECTION = None           # 예: 30.0 (deg) 방향성 있으면 / 등방성이면 None
DIR_WIDTH = 30.0           # DIRECTION 사용 시 ± 폭 (deg)
AMP = 0.10                 # wave 진폭 (실측 대비 매칭; 0.05~0.15 권장)

print(f"band = [{K_LO:.4f}, {K_HI:.4f}] cyc/px  | 파장 ~ [{1/K_HI:.1f}, {1/K_LO:.1f}] px")
print(f"direction = {DIRECTION}, amp = {AMP}")

## 4. band-pass noise wave field generator

white noise → FFT → band annulus 만 통과 → IFFT → 실수 ripple field $w(p)$ (zero-mean, ~[-1,1]).
단일 sin 이 아니라 band 전체를 덮는 비반복 패턴이라 실제 간섭무늬에 가깝고 매번 다름.

In [ ]:
def make_wave_field(shape, k_lo, k_hi, rng, direction=None, dir_width=30.0):
    H, W = shape
    noise = rng.standard_normal((H, W))
    F = np.fft.fftshift(np.fft.fft2(noise))
    fy = np.fft.fftshift(np.fft.fftfreq(H))
    fx = np.fft.fftshift(np.fft.fftfreq(W))
    KX, KY = np.meshgrid(fx, fy)
    K = np.sqrt(KX ** 2 + KY ** 2)
    mask = (K >= k_lo) & (K <= k_hi)
    if direction is not None:
        ang = (np.degrees(np.arctan2(KY, KX)) % 180)
        d = np.minimum(np.abs(ang - direction), 180 - np.abs(ang - direction))
        mask = mask & (d <= dir_width)
    field = np.real(np.fft.ifft2(np.fft.ifftshift(F * mask)))
    field -= field.mean()
    s = field.std()
    if s > 0:
        field /= s
    field = np.clip(field, -3, 3) / 3.0   # ~[-1, 1]
    return field.astype(np.float32)


field = make_wave_field(sample.shape, K_LO, K_HI, RNG, DIRECTION, DIR_WIDTH)
Pf, _, _ = power_spectrum(field)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(field, cmap="gray"); plt.title("synthetic wave field w(p)"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(np.log1p(Pf), cmap="magma"); plt.title("field power spectrum (band ring 확인)"); plt.axis("off")
plt.tight_layout(); plt.show()

## 5. wave augmentation (mean-preserving, float)

$I' = I\,(1 + a\,w)$ (mean-1 곱셈, 전역 밝기 보존) 또는 log 도메인 $\,I'=\exp(\log I + a w)$.
**float 유지·hard clip 금지** → saturation/정보 손실 없음. 학습 파이프라인에서는 이 함수를
**student global view 에만** 적용하면 됨 (teacher 미적용).

In [ ]:
def apply_wave(img, rng, k_lo, k_hi, amp, direction=None, dir_width=30.0,
               mode="mult", p=1.0, amp_jitter=True, spatial_envelope=False):
    """img(float [0,1]) 에 wave 적용. float 반환(클립 안 함).
    p: 적용 확률 (1-p 확률로 원본 그대로 → '있는/없는' 섞임 자연 처리).
    amp_jitter: 진폭을 [0, amp] 에서 랜덤 (0 포함 → no-wave 케이스).
    spatial_envelope: True 면 가우시안 envelope 으로 국소(물질 내부) wave 모사."""
    if rng.random() > p:
        return img.copy()
    a = rng.uniform(0, amp) if amp_jitter else amp
    w = make_wave_field(img.shape, k_lo, k_hi, rng, direction, dir_width)
    if spatial_envelope:
        H, W = img.shape
        yy, xx = np.mgrid[0:H, 0:W]
        cy, cx = rng.uniform(0.2, 0.8, 2) * [H, W]
        sig = rng.uniform(0.15, 0.35) * max(H, W)
        env = np.exp(-(((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sig ** 2)))
        w = w * env.astype(np.float32)
    if mode == "mult":
        out = img * (1.0 + a * w)
    elif mode == "log":
        eps = 1e-3
        out = np.exp(np.log(np.clip(img, 0, 1) + eps) + a * w) - eps
    else:
        raise ValueError(mode)
    return out.astype(np.float32)


print("apply_wave 정의 완료")

## 6. 시각 검증

(a) wave 없는 이미지에 합성 wave → ripple 생기나, (b) 전역 밝기/히스토그램 보존(saturation 없음),
(c) 합성 wave 의 FFT band 가 측정 band 와 맞나.

In [ ]:
# 검증 소스: without 그룹 있으면 그걸(깨끗), 없으면 with 첫 장.
src_path = without_paths[0] if without_paths else with_paths[0]
img = load_gray(src_path)
aug = apply_wave(img, np.random.default_rng(1), K_LO, K_HI, AMP,
                 DIRECTION, DIR_WIDTH, mode="mult", p=1.0, amp_jitter=False)

fig, ax = plt.subplots(2, 3, figsize=(13, 8))
ax[0, 0].imshow(img, cmap="gray", vmin=0, vmax=1); ax[0, 0].set_title("original"); ax[0, 0].axis("off")
ax[0, 1].imshow(np.clip(aug, 0, 1), cmap="gray", vmin=0, vmax=1); ax[0, 1].set_title(f"+ wave (amp={AMP})"); ax[0, 1].axis("off")
diff = aug - img
lim = np.abs(diff).max() + 1e-9
ax[0, 2].imshow(diff, cmap="bwr", vmin=-lim, vmax=lim); ax[0, 2].set_title("difference (added wave)"); ax[0, 2].axis("off")

ax[1, 0].hist(img.ravel(), bins=80, alpha=0.6, label="orig")
ax[1, 0].hist(aug.ravel(), bins=80, alpha=0.6, label="aug")
ax[1, 0].legend(); ax[1, 0].set_title("intensity histogram")
Po, _, _ = power_spectrum(img); Pa, _, _ = power_spectrum(aug)
ax[1, 1].imshow(np.log1p(Pa), cmap="magma"); ax[1, 1].set_title("aug power spectrum"); ax[1, 1].axis("off")
ca, pa = radial_profile(aug); co, po = radial_profile(img)
ax[1, 2].plot(co, po / (po.sum() + 1e-12), label="orig")
ax[1, 2].plot(ca, pa / (pa.sum() + 1e-12), label="aug")
ax[1, 2].axvspan(K_LO, K_HI, color="green", alpha=0.15, label="target band")
ax[1, 2].legend(); ax[1, 2].set_title("radial profile (band 매칭)"); ax[1, 2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

frac_over = float(((aug > 1) | (aug < 0)).mean())
print(f"mean: orig {img.mean():.4f} -> aug {aug.mean():.4f}  (보존되면 밝아짐 없음)")
print(f"범위 이탈 픽셀 비율: {frac_over*100:.2f}%  (float 유지 시 정보 손실 아님; uint8 변환 때만 클립)")

In [ ]:
# 여러 장 / 여러 진폭으로 한눈에 (student-only aug 가 만들 변이 분포 감 잡기)
rng = np.random.default_rng(7)
srcs = (without_paths or with_paths)[:4]
fig, ax = plt.subplots(len(srcs), 4, figsize=(13, 3.2 * len(srcs)))
ax = np.atleast_2d(ax)
for r, sp in enumerate(srcs):
    im = load_gray(sp)
    ax[r, 0].imshow(im, cmap="gray", vmin=0, vmax=1); ax[r, 0].axis("off")
    if r == 0: ax[r, 0].set_title("original")
    for c, a_ in enumerate([0.05, 0.10, 0.18], start=1):
        au = apply_wave(im, rng, K_LO, K_HI, a_, DIRECTION, DIR_WIDTH, amp_jitter=False)
        ax[r, c].imshow(np.clip(au, 0, 1), cmap="gray", vmin=0, vmax=1); ax[r, c].axis("off")
        if r == 0: ax[r, c].set_title(f"amp={a_}")
plt.tight_layout(); plt.show()

## 7. (선택) FFT notch de-wave

band 가 깨끗하게 분리되면 측정 band 를 notch 로 죽여 **실제 wave 를 제거**할 수 있다.
→ wave 있는 이미지의 '깨끗한 teacher anchor' 생성용(upgrade). 단 band 가 구조 edge 와 겹치면
구조도 깎이므로 결과를 눈으로 검증할 것.

In [ ]:
def notch_dewave(img, k_lo, k_hi, direction=None, dir_width=30.0, soft=0.0):
    H, W = img.shape
    F = np.fft.fftshift(np.fft.fft2(img))
    fy = np.fft.fftshift(np.fft.fftfreq(H)); fx = np.fft.fftshift(np.fft.fftfreq(W))
    KX, KY = np.meshgrid(fx, fy); K = np.sqrt(KX ** 2 + KY ** 2)
    notch = (K >= k_lo) & (K <= k_hi)
    if direction is not None:
        ang = (np.degrees(np.arctan2(KY, KX)) % 180)
        d = np.minimum(np.abs(ang - direction), 180 - np.abs(ang - direction))
        notch = notch & (d <= dir_width)
    gain = np.ones_like(K); gain[notch] = soft   # soft=0 완전제거, 0<soft<1 약화
    out = np.real(np.fft.ifft2(np.fft.ifftshift(F * gain)))
    return out.astype(np.float32)


wsrc = load_gray(with_paths[0])
dew = notch_dewave(wsrc, K_LO, K_HI, DIRECTION, DIR_WIDTH, soft=0.0)
fig, ax = plt.subplots(1, 3, figsize=(13, 4.3))
ax[0].imshow(wsrc, cmap="gray", vmin=0, vmax=1); ax[0].set_title("with wave (orig)"); ax[0].axis("off")
ax[1].imshow(np.clip(dew, 0, 1), cmap="gray", vmin=0, vmax=1); ax[1].set_title("notch de-waved"); ax[1].axis("off")
dd = wsrc - dew; lim = np.abs(dd).max() + 1e-9
ax[2].imshow(dd, cmap="bwr", vmin=-lim, vmax=lim); ax[2].set_title("removed (wave?)"); ax[2].axis("off")
plt.tight_layout(); plt.show()
print("removed 가 ripple 패턴이면 성공. 구조 edge 가 보이면 band 가 너무 넓음 → 줄일 것.")

---
## 학습 파이프라인 연결 (다음 단계)

위에서 확정한 `K_LO, K_HI, DIRECTION, AMP` 와 `make_wave_field` / `apply_wave` 로직을
`dino_v3/dinov3/data/augmentations.py` 에 `WaveModulation` 으로 옮겨, **student global view 에만**
적용 (teacher pipeline 미적용). 이때:
- 입력은 float, `apply_wave(mode='mult', p=0.5, amp_jitter=True)` → '있는/없는' 자연 혼합
- gamma 전역 aug 는 계속 off 유지 (물질 DC step 보존)
- 의도: PCA 가 보여준 반사 과반응 spurious 축 제거 → weak sup 의 per-patch 신호 정제(enabler)

근거/한계: `docs/ANALYSIS_grayscale_separation.md` §6–7 (asymmetric wave invariance, Pareto).